# Practical AgentOps: From PoC to Prod with MLflow 3
## 2 · MLflow Setup

**ODSC AI East 2026 · Boston · April 28, 2026**

---

This notebook walks you through the one-time environment setup for the workshop. By the end you will have:

- MLflow, OpenAI SDK, and OpenAI Agents SDK installed in your `uv` virtual environment
- A local MLflow tracking server running against a SQLite database
- A `.env` file holding your Gemini API key
- A verified connection between Python, Gemini (via the OpenAI-compatible API), and MLflow

> **Run these cells once, then leave the MLflow server running for the rest of the workshop.**

## Step 1 — Install Dependencies

Run these in a **terminal** (not inside the notebook), with your virtual environment activated.

We're installing three packages:
- **`mlflow`** — experiment tracking, tracing, and evaluation
- **`openai`** — OpenAI's official Python SDK for chat completions and embeddings
- **`openai-agents`** — OpenAI's agents SDK for building multi-step, tool-using agents
- **`python-dotenv`** — loads `.env` files into environment variables

In [ ]:
# Run in terminal — adds packages to your uv project
# uv add mlflow openai openai-agents python-dotenv

You can also add them individually — useful if you want to pin a specific version:

```bash
uv add mlflow
uv add openai
uv add openai-agents
uv add python-dotenv
```

After running these, `uv` updates both `pyproject.toml` (the dependency list) and `uv.lock` (the pinned versions). Commit both files to git so teammates get the same environment.

### Verify installation

In [ ]:
import mlflow
import openai
import agents  # openai-agents
import dotenv

print(f"mlflow        : {mlflow.__version__}")
print(f"openai        : {openai.__version__}")
print(f"openai-agents : {agents.__version__}")
print(f"python-dotenv : {dotenv.__version__}")

---

## Step 2 — Start the MLflow Tracking Server

Open a **new terminal tab**, activate your venv, and run:

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Leave that terminal running for the duration of the workshop. The UI will be available at **http://127.0.0.1:5000**.

> **Why SQLite?** The default filesystem backend (`./mlruns`) is being deprecated in MLflow 3. SQLite gives you a proper database backend with full feature support.

In [ ]:
# Verify the MLflow server is reachable
import urllib.request

try:
    urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=3)
    print("✅ MLflow server is running at http://127.0.0.1:5000")
except Exception:
    print("❌ MLflow server is NOT reachable.")
    print("   Start it in a terminal with:")
    print("   mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")

---

## Step 3 — Create a `.env` File

Your `.env` file stores secrets (API keys) outside of your code. It lives at the **root of your project** and is never committed to git.

### 3a. Create the file

Create a file named `.env` in your project root with the following content:

```
# .env — do NOT commit this file to git

# Gemini API key — get yours at https://aistudio.google.com/apikey
GEMINI_API_KEY=your-api-key-here

# Other providers — uncomment whichever you're using instead
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...

# MLflow tracking server
MLFLOW_TRACKING_URI=http://127.0.0.1:5000
```

> **How it works:** The OpenAI SDK supports any OpenAI-compatible endpoint. We point it at Google's Gemini API (`generativelanguage.googleapis.com`) using our `GEMINI_API_KEY`, so we get the OpenAI SDK's ergonomics (and MLflow's OpenAI autologging) with Gemini models underneath.

### 3b. Make sure `.env` is gitignored

Check your `.gitignore` contains `.env` before doing anything else.

In [ ]:
import os

# Check that .env is listed in .gitignore
gitignore_path = os.path.join(os.path.dirname(os.getcwd()), ".gitignore")
if not os.path.exists(gitignore_path):
    gitignore_path = os.path.join(os.getcwd(), ".gitignore")

if os.path.exists(gitignore_path):
    with open(gitignore_path) as f:
        contents = f.read()
    if ".env" in contents:
        print("✅ .gitignore contains .env — your secrets are protected")
    else:
        print("⚠️  WARNING: .env is NOT in .gitignore — add it before committing!")
else:
    print("⚠️  No .gitignore found — create one and add .env to it")

---

## Step 4 — Verify Your API Key

### 4a. Load the `.env` file and check the key exists

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env from the current working directory (or parent dirs)

REQUIRED_KEYS = {
    "GEMINI_API_KEY": "https://aistudio.google.com/apikey",
    # "OPENAI_API_KEY": "https://platform.openai.com/api-keys",
    # "ANTHROPIC_API_KEY": "https://console.anthropic.com/settings/keys",
}

all_ok = True
for key, url in REQUIRED_KEYS.items():
    value = os.getenv(key)
    if not value:
        print(f"❌ {key} is not set. Get your key at: {url}")
        all_ok = False
    elif len(value) < 10:
        print(f"⚠️  {key} looks too short — double-check it's correct")
        all_ok = False
    else:
        print(f"✅ {key} is set")

if all_ok:
    print("\n✅ All required API keys found in environment")

### 4b. Make a test LLM call

Confirm your key works end-to-end by hitting the API.

In [ ]:
from openai import OpenAI

# Point the OpenAI SDK at Gemini's OpenAI-compatible endpoint
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# To use native OpenAI instead, swap to:
# client = OpenAI()  # reads OPENAI_API_KEY automatically

try:
    response = client.chat.completions.create(
        model="gemini-2.0-flash",
        messages=[{"role": "user", "content": "Reply with exactly: 'API key works!'"}],
        max_tokens=20,
    )
    reply = response.choices[0].message.content.strip()
    print(f"✅ Gemini responded: {reply}")
except Exception as e:
    print(f"❌ API call failed: {e}")
    print("\nCommon causes:")
    print("  - GEMINI_API_KEY is wrong or not loaded (re-run the cell above)")
    print("  - Network issue")

### 4c. Verify MLflow is connected and can log a trace

In [ ]:
import mlflow
from openai import OpenAI

tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("workshop-setup-check")
mlflow.openai.autolog()  # traces every OpenAI SDK call, regardless of backend

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# To use native OpenAI instead:
# client = OpenAI()

@mlflow.trace
def setup_test_call(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gemini-2.0-flash",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
    )
    return response.choices[0].message.content.strip()

try:
    result = setup_test_call("Say 'MLflow setup complete!' and nothing else.")
    print(f"✅ MLflow trace logged successfully")
    print(f"   Response: {result}")
    print(f"\n   View it in the UI: {tracking_uri}")
except Exception as e:
    print(f"❌ Failed to log trace: {e}")

---

## Setup Complete ✅

If all the checks above passed, you're ready for the workshop. Here's a summary of what's running:

| Component | Value |
|-----------|-------|
| MLflow UI | http://127.0.0.1:5000 |
| Tracking backend | `sqlite:///mlflow.db` |
| LLM provider | Gemini via OpenAI-compatible API |
| Key variable | `GEMINI_API_KEY` in `.env` |

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `❌ MLflow server is NOT reachable` | Run `mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000` in a terminal |
| `❌ GEMINI_API_KEY is not set` | Edit `.env` and paste your key — make sure there are no spaces around `=` |
| `❌ API call failed: AuthenticationError` | Key is set but invalid — regenerate it at aistudio.google.com |
| `ModuleNotFoundError` | Run `uv add mlflow openai openai-agents python-dotenv` in a terminal |